In [1]:
import pandas as pd

In [ ]:
df = pd.read_excel('data/online_retail_II.xlsx')
# If you aren't sure which ones, this converts all object-type columns
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str)
# Save as parquet
df.to_parquet('data/online_retail_II.parquet', engine='pyarrow', index=False)

In [8]:
# 3. Read from the Parquet file for future operations
df = pd.read_parquet('data/online_retail_II.parquet', engine='pyarrow')

# Verify the data
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
df.shape

(525461, 8)

In [ ]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [ ]:
import duckdb

# Query directly using the variable name
result = duckdb.sql("SELECT country, COUNT(*) as count FROM df GROUP BY country ORDER BY count DESC LIMIT 5").df()
print(result)

          Country   count
0  United Kingdom  485852
1            EIRE    9670
2         Germany    8129
3          France    5772
4     Netherlands    2769


In [4]:
df['CustomerID'] = df['Customer ID']

In [17]:
sql = """
SELECT invoice, COUNT(*) as total
FROM df
GROUP BY invoice
ORDER BY total DESC
LIMIT 5
"""

In [18]:
result = duckdb.sql(sql).df()
print(result)

  Invoice  total
0  537434    675
1  538071    652
2  537638    601
3  537237    597
4  536876    593


In [2]:
import pandas as pd
import requests

start_date='2009-01-01'
end_date='2009-12-31'

url = f"https://api.frankfurter.app/{start_date}..{end_date}?from=GBP&to=USD"
print(f"Fetching exchange rates from {url}")

response = requests.get(url)
response.raise_for_status()
data = response.json()

# Parse rates into a flat list
rates_data = []
for date_str, rates in data.get("rates", {}).items():
    rates_data.append({
        "date": date_str,
        "gbp_to_usd": rates.get("USD")
    })
    
df = pd.DataFrame(rates_data)

Fetching exchange rates from https://api.frankfurter.app/2009-01-01..2009-12-31?from=GBP&to=USD


In [9]:
df['date'] = pd.to_datetime(df['date'])
df.head()

,date,gbp_to_usd
0,2008-12-31,1.4611
1,2009-01-02,1.4429
2,2009-01-05,1.4514
3,2009-01-06,1.4529
4,2009-01-07,1.5034


In [74]:
import duckdb
# 1. Register the DataFrame and create the view
duckdb.sql("""
CREATE VIEW dated_df AS
SELECT YEAR(date) AS year, MONTHNAME(date) as month, DAY(date) AS day, gbp_to_usd
FROM df
""")

In [75]:
sql = """
select *
from dated_df
"""

In [76]:
result=duckdb.sql(sql).df()
print(result)

     year     month  day  gbp_to_usd
0    2008  December   31      1.4611
1    2009   January    2      1.4429
2    2009   January    5      1.4514
3    2009   January    6      1.4529
4    2009   January    7      1.5034
..    ...       ...  ...         ...
252  2009  December   24      1.5984
253  2009  December   28      1.5980
254  2009  December   29      1.5989
255  2009  December   30      1.5861
256  2009  December   31      1.6221

[257 rows x 4 columns]
